# Cosmic Ray — Mutation Testing
Install: `pip install cosmic-ray` | CLI: `cosmic-ray init cfg.toml session.sqlite && cosmic-ray exec cfg.toml session.sqlite && cr-report session.sqlite`

In [1]:
import cosmic_ray
print(dir(cosmic_ray))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


In [2]:
import pkgutil, importlib
for finder, name, ispkg in pkgutil.walk_packages(cosmic_ray.__path__, cosmic_ray.__name__ + '.', onerror=lambda x: None):
    try:
        mod = importlib.import_module(name)
        print(name, '->', dir(mod))
    except Exception as e:
        print(name, '-> SKIP:', e)

cosmic_ray.ast -> ['ABC', 'Path', 'Visitor', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'abstractmethod', 'ast_nodes', 'dump_node', 'get_ast', 'get_ast_from_path', 'io', 'is_none', 'is_number', 'parso', 'read_python_source']
cosmic_ray.ast.ast_query -> ['ASTQuery', 'ASTQueryOptional', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'parso']


cosmic_ray.cli -> ['ExitCode', 'MutationSpec', 'PackageNotFoundError', 'Path', 'RichHandler', 'TestOutcome', 'WorkDB', 'WorkItem', '_SIGNAL_EXIT_CODE_BASE', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '__version__', 'apply', 'apply_mutation', 'asdict', 'baseline', 'cli', 'click', 'contextmanager', 'cosmic_ray', 'dataclasses', 'defaultdict', 'distributors', 'dump', 'handle_exec', 'http_worker', 'init', 'json', 'load_config', 'log', 'logging', 'main', 'mutate_and_test', 'new_config', 'operators', 'os', 'redirect_stdout', 'report_progress', 'serialize_config', 'signal', 'subprocess', 'sys', 'tempfile', 'use_db', 'version']
cosmic_ray.commands -> ['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'execute', 'init', 'new_config']
cosmic_ray.commands.execute -> ['ConfigDict', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package

cosmic_ray.tools.http_workers -> ['LOCALHOST_ADDRESSES', 'Path', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_create_clone', '_urls_to_args', 'asyncio', 'click', 'contextlib', 'cosmic_ray', 'git', 'log', 'logging', 'main', 'run', 'shutil', 'tempfile', 'yarl']
cosmic_ray.tools.report -> ['WorkDB', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'click', 'display_work_item', 'kills_count', 'report', 'survival_rate', 'use_db']
cosmic_ray.tools.survival_rate -> ['SUPPORTED_Z_SCORES', 'WorkDB', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'click', 'format_survival_rate', 'kills_count', 'math', 'survival_rate', 'sys', 'use_db']
cosmic_ray.tools.xml -> ['TestOutcome', 'WorkDB', 'WorkerOutcome', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_create_elemen

In [3]:
# Set up and run a mutation session, then emit raw report
import subprocess, tempfile, os

proj = tempfile.mkdtemp()

with open(os.path.join(proj, 'calculator.py'), 'w') as f:
    f.write('''
def add(a, b): return a + b
def is_positive(n): return n > 0
def safe_divide(a, b):
    if b == 0: return None
    return a / b
''')

with open(os.path.join(proj, 'test_calculator.py'), 'w') as f:
    f.write('''
from calculator import add, is_positive, safe_divide
import pytest
def test_add(): assert add(2, 3) == 5; assert add(-1, 1) == 0
def test_is_positive(): assert is_positive(1); assert not is_positive(-1)
def test_safe_divide(): assert safe_divide(10, 2) == 5.0; assert safe_divide(5, 0) is None
''')

proj_fwd = proj.replace('\\', '/')
cfg = f'''
[cosmic-ray]
module-path = "calculator.py"
timeout = 10.0
test-command = "python -m pytest {proj_fwd}/test_calculator.py -x -q"

[cosmic-ray.distributor]
name = "local"
'''

cfg_path = os.path.join(proj, 'cr.toml')
session = os.path.join(proj, 'session.sqlite')
with open(cfg_path, 'w') as f:
    f.write(cfg)

# Init
r1 = subprocess.run(['cosmic-ray', 'init', cfg_path, session],
                    capture_output=True, text=True, encoding='utf-8', errors='replace', cwd=proj, timeout=30)
print('INIT STDOUT:', r1.stdout)
print('INIT STDERR:', r1.stderr)
print('INIT CODE:', r1.returncode)

INIT STDOUT: 
INIT STDERR: 
INIT CODE: 0


In [4]:
# Execute mutation tests
r2 = subprocess.run(['cosmic-ray', 'exec', cfg_path, session],
                    capture_output=True, text=True, encoding='utf-8', errors='replace', cwd=proj, timeout=300)
print('EXEC STDOUT:', r2.stdout)
print('EXEC STDERR:', r2.stderr)
print('EXEC CODE:', r2.returncode)

EXEC STDOUT: 
EXEC STDERR: 
EXEC CODE: 0


In [5]:
# Raw cr-report text output
r3 = subprocess.run(['cr-report', session], capture_output=True, text=True, encoding='utf-8', errors='replace', timeout=30)
print(r3.stdout)
print(r3.stderr)

[job-id] f7cf7d3845b5461d9ccce3df4d330ea5
calculator.py core/ReplaceBinaryOperator_Add_Sub 0
worker outcome: WorkerOutcome.NORMAL, test outcome: TestOutcome.KILLED
[job-id] 6cf61ba95f9340f799998b6ea12783ff
calculator.py core/ReplaceBinaryOperator_Add_Mul 0
worker outcome: WorkerOutcome.NORMAL, test outcome: TestOutcome.KILLED
[job-id] 3fa90edb33db496f917ae1af69180a23
calculator.py core/ReplaceBinaryOperator_Add_Div 0
worker outcome: WorkerOutcome.NORMAL, test outcome: TestOutcome.KILLED
[job-id] c8c9d51fc3c149c69c17291ba47d4a24
calculator.py core/ReplaceBinaryOperator_Add_FloorDiv 0
worker outcome: WorkerOutcome.NORMAL, test outcome: TestOutcome.KILLED
[job-id] c496b12bae824efc999430de01d36411
calculator.py core/ReplaceBinaryOperator_Add_Mod 0
worker outcome: WorkerOutcome.NORMAL, test outcome: TestOutcome.KILLED
[job-id] 5895cae2948a4eff8247f2c0323ed346
calculator.py core/ReplaceBinaryOperator_Add_Pow 0
worker outcome: WorkerOutcome.NORMAL, test outcome: TestOutcome.KILLED
[job-id] 35

In [6]:
# Raw session SQLite data — all work items
import sqlite3, json

if os.path.exists(session):
    con = sqlite3.connect(session)
    cur = con.cursor()
    tables = cur.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    print('Tables:', tables)
    for (tbl,) in tables:
        cols = [d[0] for d in cur.execute(f'PRAGMA table_info({tbl})').fetchall()]
        rows = cur.execute(f'SELECT * FROM {tbl}').fetchall()
        records = [dict(zip(cols, row)) for row in rows]
        print(f'\n--- {tbl} ({len(records)} rows) ---')
        print(json.dumps(records, default=str, indent=2))
    con.close()

Tables: [('work_items',), ('mutation_specs',), ('work_results',)]

--- work_items (37 rows) ---
[
  {
    "0": "f7cf7d3845b5461d9ccce3df4d330ea5"
  },
  {
    "0": "6cf61ba95f9340f799998b6ea12783ff"
  },
  {
    "0": "3fa90edb33db496f917ae1af69180a23"
  },
  {
    "0": "c8c9d51fc3c149c69c17291ba47d4a24"
  },
  {
    "0": "c496b12bae824efc999430de01d36411"
  },
  {
    "0": "5895cae2948a4eff8247f2c0323ed346"
  },
  {
    "0": "3534c4f75e614a2fadcf552338caba67"
  },
  {
    "0": "11d747f393524fd88b5734052508d96b"
  },
  {
    "0": "aeef2e302ee34ab1b6bd2510f33cc502"
  },
  {
    "0": "03f899542fd44276a39df43a4559a4ed"
  },
  {
    "0": "cc58b64b37a2421a9359b38a246f72db"
  },
  {
    "0": "35111103f3174659abf530d57af844c8"
  },
  {
    "0": "c741cac21f6b4a5b9b0394c3cbb8c3fb"
  },
  {
    "0": "f4e9e0f8a8c740da894bcd9f887bc6d8"
  },
  {
    "0": "6a41097ec6124a8ea22fa61cc4047335"
  },
  {
    "0": "8ca0642e24ac44f0a1725f6775602142"
  },
  {
    "0": "b388d41b9c7d4b12b89920f4d3f8615b"
  },
 